# 🚀 Reportes DAO — Demo y pruebas
**Proyecto:** CivicTech  
**Objetivo:** demostrar creación, consulta y gestión de reportes con evidencias usando `ReportesDAO`.  
**Entorno:** Jupyter Notebook · Python · MongoDB (ajustar `MONGO_URI` en la celda de conexión)  

**Instrucciones rápidas**
- Activá el entorno virtual: `source .venv/bin/activate`  
- Asegurate de que MongoDB esté accesible desde `MONGO_URI`.  
- Si la importación falla, usá `from dao.reporte_dao import ReportesDAO` o añadí `from .reporte_dao import ReportesDAO` en `dao/__init__.py`.


In [1]:
# Celda: inicializar MongoDAO y dejar `mongo` disponible
from dao.mongo_dao import MongoDAO
mongo = MongoDAO()
db = mongo.connect()
print("Conectado a DB:", mongo.db.name)

Conectado a DB: civictech


In [2]:
# Celda: inicializar ReportesDAO (ejecutá solo esta)
from dao.reporte_dao import ReportesDAO
import pprint
pp = pprint.PrettyPrinter(indent=2)

reportes_dao = ReportesDAO(mongo)
print("DAO inicializado. Colección reporte:", reportes_dao.reporte_col.name)



DAO inicializado. Colección reporte: reporte


In [3]:
# Celda 3 corregida: asegurar que `mongo` esté definido y conectado (ejecutá solo esta)
from dao.mongo_dao import MongoDAO

try:
    mongo  # si ya existe, no hacemos nada
except NameError:
    mongo = MongoDAO()
    db = mongo.connect()
    print("Conectado a DB:", mongo.db.name)
else:
    # comparar explícitamente con None para evitar NotImplementedError
    if getattr(mongo, "db", None) is None:
        db = mongo.connect()
        print("Conectado a DB:", mongo.db.name)
    else:
        # mongo.db existe; mostrar su nombre sin evaluar su truthiness
        try:
            print("Variable 'mongo' ya definida. DB:", mongo.db.name)
        except Exception as e:
            print("Variable 'mongo' definida pero no se pudo leer mongo.db.name:", type(e).__name__, str(e))


Variable 'mongo' ya definida. DB: civictech


In [9]:
# Celda: insertar un reporte seguro (timezone-aware, validación local mínima, salida compacta)
from datetime import datetime, timezone
from bson import ObjectId
from pymongo.errors import WriteError

def now_utc():
    return datetime.now(timezone.utc)

def safe_str(x, maxlen=80):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

# Ajustá la URL si querés otra imagen de ejemplo
foto_url = "https://gcdn.emol.cl/argentina/files/2016/05/auto-tapando-rampa.jpg"

# Tomar primer usuario disponible (ya tenés usuarios listados)
usuario = mongo.db.usuario.find_one({}, {"_id":1, "municipio_id":1})
if not usuario:
    raise RuntimeError("No se encontró ningún usuario en la colección 'usuario'.")

usuario_id = usuario["_id"]
municipio_id = usuario.get("municipio_id")
if not municipio_id:
    raise RuntimeError("El usuario seleccionado no tiene 'municipio_id' asignado.")

now = now_utc()

reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": municipio_id,
    "patente_vehiculo": "DEMO123",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type": "Point", "coordinates": [-67.49, -29.16]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_demo_001",
    "descripcion": "Vehículo bloqueando rampa de acceso peatonal. Foto adjunta como evidencia."
}

evidencia_doc = {
    "url_foto": foto_url,
    "hash_seguridad_sha": "sha256:demo_001",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

# Validación local mínima
def validar_reporte_local(r):
    if not isinstance(r.get("usuario_id"), ObjectId): return "usuario_id debe ser ObjectId"
    if not isinstance(r.get("municipio_id"), ObjectId): return "municipio_id debe ser ObjectId"
    if "ubicacion" not in r or not isinstance(r["ubicacion"], dict): return "ubicacion debe ser GeoJSON Point"
    if r.get("estado") not in ("Pendiente", "Validada", "Rechazada"): return "estado inválido"
    return None

def validar_evidencia_local(e):
    if "url_foto" not in e or not isinstance(e["url_foto"], str) or not e["url_foto"].strip():
        return "Falta url_foto o no es string válido"
    if "hash_seguridad_sha" not in e or not isinstance(e["hash_seguridad_sha"], str) or not e["hash_seguridad_sha"].strip():
        return "Falta hash_seguridad_sha o no es string válido"
    return None

err = validar_reporte_local(reporte_doc) or validar_evidencia_local(evidencia_doc)
if err:
    raise ValueError("Validación local falló: " + err)

# Insertar sin transacción (entorno standalone)
try:
    rres = mongo.db.reporte.insert_one(reporte_doc)
    evidencia_doc["reporte_id"] = rres.inserted_id
    eres = mongo.db.evidencia.insert_one(evidencia_doc)
except WriteError as we:
    details = getattr(we, "details", None)
    print("WriteError (validación) devuelto por MongoDB:")
    print(details if details else str(we))
    raise
except Exception as e:
    print("Error al insertar:", type(e).__name__, str(e))
    raise

# Mostrar resumen compacto incluyendo nombre del municipio
rpt = mongo.db.reporte.find_one({"_id": rres.inserted_id}, {"descripcion":1, "patente_vehiculo":1, "estado":1, "fechaHora_server":1, "usuario_id":1, "municipio_id":1, "ubicacion":1})
ev = mongo.db.evidencia.find_one({"reporte_id": rres.inserted_id}, {"url_foto":1, "hash_seguridad_sha":1, "uploaded_by":1, "uploaded_at":1})

# Recuperar nombre del municipio para salida legible
mun = mongo.db.municipio.find_one({"_id": rpt.get("municipio_id")}, {"nombre":1})
municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"

print("== Resumen del reporte insertado ==")
print("reporte_id :", safe_str(rpt.get("_id")))
print("patente    :", safe_str(rpt.get("patente_vehiculo")))
print("estado     :", safe_str(rpt.get("estado")))
print("fechaHora  :", safe_str(rpt.get("fechaHora_server")))
print("usuario_id :", safe_str(rpt.get("usuario_id")))
print("municipio  :", safe_str(rpt.get("municipio_id")), "| nombre:", safe_str(municipio_nombre))
print("ubicacion  :", safe_str(rpt.get("ubicacion")))
print("\n== Evidencia asociada ==")
print("evidencia_id:", safe_str(ev.get("_id")))
print("url_foto    :", safe_str(ev.get("url_foto")))
print("hash        :", safe_str(ev.get("hash_seguridad_sha")))
print("uploaded_by :", safe_str(ev.get("uploaded_by")))
print("uploaded_at :", safe_str(ev.get("uploaded_at")))



== Resumen del reporte insertado ==
reporte_id : 6a236a7002b1dea021ff1126
patente    : DEMO123
estado     : Pendiente
fechaHora  : 2026-06-06 00:31:44.742000
usuario_id : 6a21d09b4076ce51229df8a6
municipio  : 6a21d09b4076ce51229df8a4 | nombre: Chilecito
ubicacion  : {'type': 'Point', 'coordinates': [-67.49, -29.16]}

== Evidencia asociada ==
evidencia_id: 6a236a7002b1dea021ff1127
url_foto    : https://gcdn.emol.cl/argentina/files/2016/05/auto-tapando-rampa.jpg
hash        : sha256:demo_001
uploaded_by : 6a21d09b4076ce51229df8a6
uploaded_at : 2026-06-06 00:31:44.742000


In [11]:
# Celda siguiente (ejecutá solo esta) — insertar segundo reporte con usuario y municipio distintos
from datetime import datetime, timezone
from bson import ObjectId
from pymongo.errors import WriteError

def now_utc():
    return datetime.now(timezone.utc)

def safe_str(x, maxlen=80):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

# --- Datos: usar usuario y municipio distintos al primer reporte ---
# Usuario: Marta Silva (La Rioja Capital)
usuario_id = ObjectId("6a21d09b4076ce51229df8ab")
municipio_id = ObjectId("6a21d09b4076ce51229df8a5")

# URL de evidencia (segundo ejemplo que diste)
foto_url = "https://fotos.perfil.com/2020/05/07/autos-mal-estacionados-20200507-952139.jpg"

now = now_utc()

reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": municipio_id,
    "patente_vehiculo": "LRJ-DEM-02",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type": "Point", "coordinates": [-66.851, -29.411]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_lrj_demo_02",
    "descripcion": "Auto en doble fila frente a escuela, obstaculiza paso peatonal. Evidencia adjunta."
}

evidencia_doc = {
    "url_foto": foto_url,
    "hash_seguridad_sha": "sha256:perfil_demo_02",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

# Validación local mínima
def validar_reporte_local(r):
    if not isinstance(r.get("usuario_id"), ObjectId): return "usuario_id debe ser ObjectId"
    if not isinstance(r.get("municipio_id"), ObjectId): return "municipio_id debe ser ObjectId"
    if "ubicacion" not in r or not isinstance(r["ubicacion"], dict): return "ubicacion debe ser GeoJSON Point"
    if r.get("estado") not in ("Pendiente", "Validada", "Rechazada"): return "estado inválido"
    return None

def validar_evidencia_local(e):
    if "url_foto" not in e or not isinstance(e["url_foto"], str) or not e["url_foto"].strip():
        return "Falta url_foto o no es string válido"
    if "hash_seguridad_sha" not in e or not isinstance(e["hash_seguridad_sha"], str) or not e["hash_seguridad_sha"].strip():
        return "Falta hash_seguridad_sha o no es string válido"
    return None

err = validar_reporte_local(reporte_doc) or validar_evidencia_local(evidencia_doc)
if err:
    raise ValueError("Validación local falló: " + err)

# Insertar sin transacción (entorno standalone)
try:
    rres = mongo.db.reporte.insert_one(reporte_doc)
    evidencia_doc["reporte_id"] = rres.inserted_id
    eres = mongo.db.evidencia.insert_one(evidencia_doc)
except WriteError as we:
    details = getattr(we, "details", None)
    print("WriteError (validación) devuelto por MongoDB:")
    print(details if details else str(we))
    raise
except Exception as e:
    print("Error al insertar:", type(e).__name__, str(e))
    raise

# Mostrar resumen compacto incluyendo nombre del municipio
rpt = mongo.db.reporte.find_one({"_id": rres.inserted_id}, {"descripcion":1, "patente_vehiculo":1, "estado":1, "fechaHora_server":1, "usuario_id":1, "municipio_id":1, "ubicacion":1})
ev = mongo.db.evidencia.find_one({"reporte_id": rres.inserted_id}, {"url_foto":1, "hash_seguridad_sha":1, "uploaded_by":1, "uploaded_at":1})

# Recuperar nombre del municipio para salida legible
mun = mongo.db.municipio.find_one({"_id": rpt.get("municipio_id")}, {"nombre":1})
municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"

print("== Resumen del reporte insertado ==")
print("reporte_id :", safe_str(rpt.get("_id")))
print("patente    :", safe_str(rpt.get("patente_vehiculo")))
print("estado     :", safe_str(rpt.get("estado")))
print("fechaHora  :", safe_str(rpt.get("fechaHora_server")))
print("usuario_id :", safe_str(rpt.get("usuario_id")))
print("municipio  :", safe_str(rpt.get("municipio_id")), "| nombre:", safe_str(municipio_nombre))
print("ubicacion  :", safe_str(rpt.get("ubicacion")))
print("\n== Evidencia asociada ==")
print("evidencia_id:", safe_str(ev.get("_id")))
print("url_foto    :", safe_str(ev.get("url_foto")))
print("hash        :", safe_str(ev.get("hash_seguridad_sha")))
print("uploaded_by :", safe_str(ev.get("uploaded_by")))
print("uploaded_at :", safe_str(ev.get("uploaded_at")))


== Resumen del reporte insertado ==
reporte_id : 6a236abf02b1dea021ff112a
patente    : LRJ-DEM-02
estado     : Pendiente
fechaHora  : 2026-06-06 00:33:03.868000
usuario_id : 6a21d09b4076ce51229df8ab
municipio  : 6a21d09b4076ce51229df8a5 | nombre: La Rioja Capital
ubicacion  : {'type': 'Point', 'coordinates': [-66.851, -29.411]}

== Evidencia asociada ==
evidencia_id: 6a236abf02b1dea021ff112b
url_foto    : https://fotos.perfil.com/2020/05/07/autos-mal-estacionados-20200507-952139.jpg
hash        : sha256:perfil_demo_02
uploaded_by : 6a21d09b4076ce51229df8ab
uploaded_at : 2026-06-06 00:33:03.868000


In [16]:
# Celda: insertar reporte seguro y mostrar nombre del municipio en la salida (ejecutá solo esta)
from datetime import datetime, timezone
from bson import ObjectId
from pymongo.errors import WriteError

def now_utc():
    return datetime.now(timezone.utc)

def safe_str(x, maxlen=120):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

# --- Ajustá estos valores si querés otro usuario/municipio ---
usuario_id = ObjectId("6a21d09b4076ce51229df8ab")   # ejemplo: Marta Silva
municipio_id = ObjectId("6a21d09b4076ce51229df8a5") # ejemplo: La Rioja Capital

# URL de evidencia (ejemplo)
foto_url = "https://fotos.perfil.com/2020/05/07/autos-mal-estacionados-20200507-952139.jpg"

now = now_utc()

reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": municipio_id,
    "patente_vehiculo": "LRJ-DEM-02",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type": "Point", "coordinates": [-66.851, -29.411]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_lrj_demo_02",
    "descripcion": "Auto en doble fila frente a escuela, obstaculiza paso peatonal. Evidencia adjunta."
}

evidencia_doc = {
    "url_foto": foto_url,
    "hash_seguridad_sha": "sha256:perfil_demo_02",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

# Validación local mínima
def validar_reporte_local(r):
    if not isinstance(r.get("usuario_id"), ObjectId): return "usuario_id debe ser ObjectId"
    if not isinstance(r.get("municipio_id"), ObjectId): return "municipio_id debe ser ObjectId"
    if "ubicacion" not in r or not isinstance(r["ubicacion"], dict): return "ubicacion debe ser GeoJSON Point"
    if r.get("estado") not in ("Pendiente", "Validada", "Rechazada"): return "estado inválido"
    return None

def validar_evidencia_local(e):
    if "url_foto" not in e or not isinstance(e["url_foto"], str) or not e["url_foto"].strip():
        return "Falta url_foto o no es string válido"
    if "hash_seguridad_sha" not in e or not isinstance(e["hash_seguridad_sha"], str) or not e["hash_seguridad_sha"].strip():
        return "Falta hash_seguridad_sha o no es string válido"
    return None

err = validar_reporte_local(reporte_doc) or validar_evidencia_local(evidencia_doc)
if err:
    raise ValueError("Validación local falló: " + err)

# Insertar sin transacción (entorno standalone)
try:
    rres = mongo.db.reporte.insert_one(reporte_doc)
    evidencia_doc["reporte_id"] = rres.inserted_id
    eres = mongo.db.evidencia.insert_one(evidencia_doc)
except WriteError as we:
    details = getattr(we, "details", None)
    print("WriteError (validación) devuelto por MongoDB:")
    print(details if details else str(we))
    raise
except Exception as e:
    print("Error al insertar:", type(e).__name__, str(e))
    raise

# Recuperar datos para salida legible, incluyendo nombre del municipio
rpt = mongo.db.reporte.find_one({"_id": rres.inserted_id}, {"descripcion":1, "patente_vehiculo":1, "estado":1, "fechaHora_server":1, "usuario_id":1, "municipio_id":1, "ubicacion":1})
ev = mongo.db.evidencia.find_one({"reporte_id": rres.inserted_id}, {"url_foto":1, "hash_seguridad_sha":1, "uploaded_by":1, "uploaded_at":1})

# Buscar nombre del municipio
mun = mongo.db.municipio.find_one({"_id": rpt.get("municipio_id")}, {"nombre":1})
municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"

# Mostrar resumen compacto con nombre del municipio
print("== Resumen del reporte insertado ==")
print("reporte_id :", safe_str(rpt.get("_id")))
print("patente    :", safe_str(rpt.get("patente_vehiculo")))
print("estado     :", safe_str(rpt.get("estado")))
print("fechaHora  :", safe_str(rpt.get("fechaHora_server")))
print("usuario_id :", safe_str(rpt.get("usuario_id")))
print("municipio  :", safe_str(rpt.get("municipio_id")), "| nombre:", safe_str(municipio_nombre))
print("ubicacion  :", safe_str(rpt.get("ubicacion")))
print("\n== Evidencia asociada ==")
print("evidencia_id:", safe_str(ev.get("_id")))
print("url_foto    :", safe_str(ev.get("url_foto")))
print("hash        :", safe_str(ev.get("hash_seguridad_sha")))
print("uploaded_by :", safe_str(ev.get("uploaded_by")))
print("uploaded_at :", safe_str(ev.get("uploaded_at")))


== Resumen del reporte insertado ==
reporte_id : 6a236b7c02b1dea021ff112c
patente    : LRJ-DEM-02
estado     : Pendiente
fechaHora  : 2026-06-06 00:36:12.198000
usuario_id : 6a21d09b4076ce51229df8ab
municipio  : 6a21d09b4076ce51229df8a5 | nombre: La Rioja Capital
ubicacion  : {'type': 'Point', 'coordinates': [-66.851, -29.411]}

== Evidencia asociada ==
evidencia_id: 6a236b7c02b1dea021ff112d
url_foto    : https://fotos.perfil.com/2020/05/07/autos-mal-estacionados-20200507-952139.jpg
hash        : sha256:perfil_demo_02
uploaded_by : 6a21d09b4076ce51229df8ab
uploaded_at : 2026-06-06 00:36:12.198000


# Ejecutar esta celda para listar los reportes por municipio.
# Muestra la cantidad de reportes y un resumen compacto (ID corto, patente, estado, fecha, descripción).
# Útil para la presentación: verificar rápidamente dónde mirar (municipio y cantidad).
# Nota: asegúrate de haber ejecutado antes las celdas de inicialización (MongoDAO y ReportesDAO).


In [19]:
# Ejecutá solo esta celda: listar reportes por municipio con fila de encabezados
from bson import ObjectId
from datetime import datetime

MAX_REPORTS = 200

pipeline = [
    {"$sort": {"fechaHora_server": -1}},
    {"$limit": MAX_REPORTS},
    {"$lookup": {
        "from": "municipio",
        "localField": "municipio_id",
        "foreignField": "_id",
        "as": "mun"
    }},
    {"$unwind": {"path": "$mun", "preserveNullAndEmptyArrays": True}},
    {"$project": {
        "_id": 1,
        "patente_vehiculo": 1,
        "estado": 1,
        "fechaHora_server": 1,
        "usuario_id": 1,
        "municipio_id": 1,
        "municipio_nombre": "$mun.nombre",
        "descripcion": 1
    }}
]

rows = list(mongo.db.reporte.aggregate(pipeline))

def short_id(oid, n=5):
    s = str(oid) if oid is not None else ""
    return s[:n]

def fmt_date(d):
    return d.isoformat(sep=" ") if d is not None else ""

if not rows:
    print("No hay reportes en la colección.")
else:
    # Agrupar por municipio_nombre (usar 'SIN_NOMBRE' si no se encuentra)
    grouped = {}
    for r in rows:
        mun = r.get("municipio_nombre") or "SIN_NOMBRE"
        grouped.setdefault(mun, []).append(r)

    for mun, rep_list in grouped.items():
        print("="*100)
        print(f"Municipio: {mun}  —  Reportes: {len(rep_list)}")
        print("-"*100)

        # Preparar tabla con encabezados
        headers = ["ID", "Patente", "Estado", "Fecha", "Descripción"]
        # Calcular anchos (máx por columna)
        col_widths = {h: len(h) for h in headers}
        for r in rep_list:
            col_widths["ID"] = max(col_widths["ID"], len(short_id(r.get("_id"), 5)))
            col_widths["Patente"] = max(col_widths["Patente"], len(str(r.get("patente_vehiculo") or "")))
            col_widths["Estado"] = max(col_widths["Estado"], len(str(r.get("estado") or "")))
            col_widths["Fecha"] = max(col_widths["Fecha"], len(fmt_date(r.get("fechaHora_server"))))
            col_widths["Descripción"] = max(col_widths["Descripción"], min(60, len((r.get("descripcion") or "")[:120])))

        # Limitar ancho máximo razonable
        for k in col_widths:
            if col_widths[k] > 60:
                col_widths[k] = 60

        sep = " | "
        # Imprimir encabezado
        header_line = sep.join(h.ljust(col_widths[h]) for h in headers)
        divider = "-+-".join("-" * col_widths[h] for h in headers)
        print(header_line)
        print(divider)

        # Imprimir filas
        for r in rep_list:
            fid = short_id(r.get("_id"), 5).ljust(col_widths["ID"])
            patente = (r.get("patente_vehiculo") or "").ljust(col_widths["Patente"])
            estado = (r.get("estado") or "").ljust(col_widths["Estado"])
            fecha = fmt_date(r.get("fechaHora_server")).ljust(col_widths["Fecha"])
            desc = (r.get("descripcion") or "")[:col_widths["Descripción"]].ljust(col_widths["Descripción"])
            print(f"{fid} | {patente} | {estado} | {fecha} | {desc}")
        print()


Municipio: Chilecito  —  Reportes: 13
----------------------------------------------------------------------------------------------------
ID    | Patente  | Estado    | Fecha                      | Descripción                                                 
------+----------+-----------+----------------------------+-------------------------------------------------------------
6a236 | EVID-002 | Pendiente | 2026-06-06 00:37:59.830000 | Prueba: crear con evidencia (debe insertarse).              
6a236 | DEMO123  | Pendiente | 2026-06-06 00:31:44.742000 | Vehículo bloqueando rampa de acceso peatonal. Foto adjunta c
6a236 | DEMO123  | Pendiente | 2026-06-06 00:31:09.276000 | Vehículo bloqueando rampa de acceso peatonal. Foto adjunta c
6a236 | DEMO123  | Pendiente | 2026-06-06 00:07:08.356000 | Vehículo bloqueando rampa de acceso peatonal. Foto adjunta c
6a235 | DEMO123  | Pendiente | 2026-06-05 23:39:47.292000 | Vehículo bloqueando rampa de acceso peatonal. Foto adjunta c
6a235 | DEMO12

In [15]:
# Celda: intentar insertar reporte SIN evidencia -> bloquear y mostrar mensaje claro
from datetime import datetime, timezone
from bson import ObjectId
from pymongo.errors import WriteError

def now_utc():
    return datetime.now(timezone.utc)

def banner(msg, kind="ERROR"):
    line = "=" * (len(msg) + 8)
    if kind == "ERROR":
        print(f"\n{line}\n!!!  {msg}  !!!\n{line}\n")
    else:
        print(f"\n{line}\n***  {msg}  ***\n{line}\n")

def validar_reporte_minimo(r):
    if not isinstance(r.get("usuario_id"), ObjectId):
        return "usuario_id debe ser ObjectId"
    if not isinstance(r.get("municipio_id"), ObjectId):
        return "municipio_id debe ser ObjectId"
    if "ubicacion" not in r or not isinstance(r["ubicacion"], dict):
        return "ubicacion debe ser GeoJSON Point"
    if r.get("estado") not in ("Pendiente", "Validada", "Rechazada"):
        return "estado inválido"
    return None

def validar_evidencia_minima(e):
    if not e:
        return "EVIDENCIA AUSENTE"
    if "url_foto" not in e or not isinstance(e["url_foto"], str) or not e["url_foto"].strip():
        return "url_foto inválida"
    if "hash_seguridad_sha" not in e or not isinstance(e["hash_seguridad_sha"], str) or not e["hash_seguridad_sha"].strip():
        return "hash_seguridad_sha inválido"
    return None

def insertar_reporte_con_evidencia(reporte_doc, evidencia_doc):
    """
    Inserta un reporte solo si viene acompañado de evidencia válida.
    Si falta evidencia, NO inserta y muestra un mensaje destacado.
    Devuelve (reporte_id, evidencia_id) en caso de éxito, o (None, None) si falla.
    """
    # Validar reporte
    err = validar_reporte_minimo(reporte_doc)
    if err:
        banner(f"Validación de reporte falló: {err}", kind="ERROR")
        return None, None

    # Regla de negocio: exigir evidencia
    err_ev = validar_evidencia_minima(evidencia_doc)
    if err_ev:
        banner("NO SE PERMITE CREAR UN REPORTE SIN EVIDENCIA", kind="ERROR")
        print("Motivo:", err_ev)
        print("Acción requerida: adjuntar 'evidencia_doc' con 'url_foto' y 'hash_seguridad_sha'.")
        return None, None

    # Intentar insertar (manejo de errores de MongoDB)
    try:
        rres = mongo.db.reporte.insert_one(reporte_doc)
        evidencia_doc["reporte_id"] = rres.inserted_id
        eres = mongo.db.evidencia.insert_one(evidencia_doc)
    except WriteError as we:
        details = getattr(we, "details", None)
        banner("WriteError al insertar (validación MongoDB)", kind="ERROR")
        print(details if details else str(we))
        return None, None
    except Exception as e:
        banner("Error inesperado al insertar", kind="ERROR")
        print(type(e).__name__, str(e))
        return None, None

    banner("REPORTE Y EVIDENCIA INSERTADOS CON ÉXITO", kind="OK")
    print("reporte_id :", rres.inserted_id)
    print("evidencia_id:", eres.inserted_id)
    return rres.inserted_id, eres.inserted_id

# -------------------------
# Ejemplo de uso (NO ejecutar si querés solo la función):
# now = now_utc()
# reporte_demo = {
#     "usuario_id": ObjectId("6a21d09b4076ce51229df8a6"),
#     "municipio_id": ObjectId("6a21d09b4076ce51229df8a4"),
#     "patente_vehiculo": "DEMO_NO_EVID",
#     "fechaHora_dispositivo": now,
#     "fechaHora_server": now,
#     "ubicacion": {"type":"Point","coordinates":[-67.49,-29.16]},
#     "estado": "Pendiente",
#     "hash_evidencia": "hash_demo_no_ev",
#     "descripcion": "Intento de crear sin evidencia"
# }
# # llamada que debe FALLAR y mostrar cartel claro:
# insertar_reporte_con_evidencia(reporte_demo, evidencia_doc=None)
#
# # llamada que debe tener éxito (ejemplo):
# evidencia_demo = {"url_foto":"https://ejemplo/1.jpg","hash_seguridad_sha":"sha256:demo","uploaded_by":reporte_demo["usuario_id"],"uploaded_at":now}
# insertar_reporte_con_evidencia(reporte_demo, evidencia_demo)


In [20]:
# Celda 1: intento de insertar REPORTE SIN EVIDENCIA -> debe bloquearse y mostrar mensaje destacado
from datetime import datetime, timezone
from bson import ObjectId

def now_utc():
    return datetime.now(timezone.utc)

def banner(msg, kind="ERROR"):
    line = "=" * (len(msg) + 8)
    if kind == "ERROR":
        print(f"\n{line}\n!!!  {msg}  !!!\n{line}\n")
    else:
        print(f"\n{line}\n***  {msg}  ***\n{line}\n")

# Datos de ejemplo (ajustá si necesitás otro usuario/municipio)
usuario_id = ObjectId("6a21d09b4076ce51229df8a6")
municipio_id = ObjectId("6a21d09b4076ce51229df8a4")
now = now_utc()

reporte_demo = {
    "usuario_id": usuario_id,
    "municipio_id": municipio_id,
    "patente_vehiculo": "NOEV-001",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type":"Point","coordinates":[-67.49,-29.16]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_no_ev_001",
    "descripcion": "Prueba: intento crear sin evidencia (debe bloquearse)."
}

# Regla de negocio: no permitir crear reporte sin evidencia
def validar_evidencia_minima(e):
    if not e:
        return "EVIDENCIA AUSENTE"
    if "url_foto" not in e or not isinstance(e["url_foto"], str) or not e["url_foto"].strip():
        return "url_foto inválida"
    if "hash_seguridad_sha" not in e or not isinstance(e["hash_seguridad_sha"], str) or not e["hash_seguridad_sha"].strip():
        return "hash_seguridad_sha inválido"
    return None

# Intento de inserción SIN evidencia
evidencia_none = None
err_ev = validar_evidencia_minima(evidencia_none)
if err_ev:
    banner("NO SE PERMITE CREAR UN REPORTE SIN EVIDENCIA", kind="ERROR")
    print("Motivo:", err_ev)
    print("Acción requerida: adjuntar 'evidencia_doc' con 'url_foto' y 'hash_seguridad_sha'.")
else:
    # (no debería llegar aquí)
    rres = mongo.db.reporte.insert_one(reporte_demo)
    print("Inserción inesperada, reporte_id:", rres.inserted_id)



!!!  NO SE PERMITE CREAR UN REPORTE SIN EVIDENCIA  !!!

Motivo: EVIDENCIA AUSENTE
Acción requerida: adjuntar 'evidencia_doc' con 'url_foto' y 'hash_seguridad_sha'.


In [21]:
# Celda 2: insertar REPORTE con EVIDENCIA -> insertar y mostrar mensaje destacado con nombre del municipio
from datetime import datetime, timezone
from bson import ObjectId
from pymongo.errors import WriteError

def now_utc():
    return datetime.now(timezone.utc)

def banner(msg, kind="OK"):
    line = "=" * (len(msg) + 8)
    if kind == "ERROR":
        print(f"\n{line}\n!!!  {msg}  !!!\n{line}\n")
    else:
        print(f"\n{line}\n***  {msg}  ***\n{line}\n")

def safe_str(x, maxlen=120):
    s = str(x) if x is not None else ""
    return (s[:maxlen-3] + "...") if len(s) > maxlen else s

# Datos de ejemplo (ajustá si necesitás otro usuario/municipio)
usuario_id = ObjectId("6a21d09b4076ce51229df8a6")
municipio_id = ObjectId("6a21d09b4076ce51229df8a4")
now = now_utc()

reporte_doc = {
    "usuario_id": usuario_id,
    "municipio_id": municipio_id,
    "patente_vehiculo": "EVID-002",
    "fechaHora_dispositivo": now,
    "fechaHora_server": now,
    "ubicacion": {"type":"Point","coordinates":[-67.49,-29.16]},
    "estado": "Pendiente",
    "hash_evidencia": "hash_ev_002",
    "descripcion": "Prueba: crear con evidencia (debe insertarse)."
}

evidencia_doc = {
    "url_foto": "https://www.parana.gob.ar/writable/uploads/oldnews/1456235344.jpg",
    "hash_seguridad_sha": "sha256:demo_ok_002",
    "uploaded_by": usuario_id,
    "uploaded_at": now
}

# Validación mínima
def validar_reporte_minimo(r):
    if not isinstance(r.get("usuario_id"), ObjectId):
        return "usuario_id debe ser ObjectId"
    if not isinstance(r.get("municipio_id"), ObjectId):
        return "municipio_id debe ser ObjectId"
    if "ubicacion" not in r or not isinstance(r["ubicacion"], dict):
        return "ubicacion debe ser GeoJSON Point"
    if r.get("estado") not in ("Pendiente", "Validada", "Rechazada"):
        return "estado inválido"
    return None

def validar_evidencia_minima(e):
    if not e:
        return "EVIDENCIA AUSENTE"
    if "url_foto" not in e or not isinstance(e["url_foto"], str) or not e["url_foto"].strip():
        return "url_foto inválida"
    if "hash_seguridad_sha" not in e or not isinstance(e["hash_seguridad_sha"], str) or not e["hash_seguridad_sha"].strip():
        return "hash_seguridad_sha inválido"
    return None

err = validar_reporte_minimo(reporte_doc) or validar_evidencia_minima(evidencia_doc)
if err:
    banner("No se puede insertar: validación falló", kind="ERROR")
    print("Motivo:", err)
else:
    try:
        # Insertar reporte y luego evidencia referenciando reporte_id
        rres = mongo.db.reporte.insert_one(reporte_doc)
        evidencia_doc["reporte_id"] = rres.inserted_id
        eres = mongo.db.evidencia.insert_one(evidencia_doc)
    except WriteError as we:
        banner("WriteError al insertar", kind="ERROR")
        print(getattr(we, "details", str(we)))
    except Exception as e:
        banner("Error inesperado al insertar", kind="ERROR")
        print(type(e).__name__, str(e))
    else:
        # Recuperar nombre del municipio para salida
        mun = mongo.db.municipio.find_one({"_id": reporte_doc["municipio_id"]}, {"nombre":1})
        municipio_nombre = mun.get("nombre") if mun else "NOMBRE_NO_ENCONTRADO"

        banner("REPORTE Y EVIDENCIA INSERTADOS CON ÉXITO", kind="OK")
        print("reporte_id :", safe_str(rres.inserted_id))
        print("evidencia_id:", safe_str(eres.inserted_id))
        print("municipio  :", safe_str(reporte_doc["municipio_id"]), "| nombre:", safe_str(municipio_nombre))



***  REPORTE Y EVIDENCIA INSERTADOS CON ÉXITO  ***

reporte_id : 6a236e3202b1dea021ff1130
evidencia_id: 6a236e3202b1dea021ff1131
municipio  : 6a21d09b4076ce51229df8a4 | nombre: Chilecito
